In [1]:
import pandas as pd
import numpy as np
import joblib
import json

# Load the GENERAL crop dataset
df_general = pd.read_csv('features_v3_ohe_crop.csv')
df_general = df_general[df_general['crop_sugarcane'] == 0].copy()

# Load the CLEAN base data
df_base = pd.read_csv('base_data_v3_soil_corrected.csv')
df_base = df_base[df_base['crop'] != 'Sugarcane'].copy() # Match the general dataset

# 1. Create Imputed Nutrient Store
# We will use the imputation we already created in features_v3_ohe_crop.csv
# We calculate the mean for each district to use as our "Basic Mode" fallback
imputed_nutrients = df_general.groupby('district_anugul').first() # Hack to get district names...

# --- This is complex. Let's do a simpler, more robust lookup ---
# We'll create the lookups from our final feature file

# Get all district column names
dist_cols = [col for col in df_general.columns if col.startswith('district_')]

imputed_data_store = {}
for col in dist_cols:
    district_name = col.split('_', 1)[1]
    # Get the first row for this district (all imputed values are the same for that district)
    district_data = df_general[df_general[col] == 1].iloc[0]
    imputed_data_store[district_name] = {
        'mean_ph': district_data['mean_ph'],
        'mean_n': district_data['mean_n'],
        'mean_p': district_data['mean_p'],
        'mean_k': district_data['mean_k']
    }
imputed_data_store['default'] = { # Global fallback
    'mean_ph': df_general['mean_ph'].mean(),
    'mean_n': df_general['mean_n'].mean(),
    'mean_p': df_general['mean_p'].mean(),
    'mean_k': df_general['mean_k'].mean()
}
with open('imputed_data_store.json', 'w') as f:
    json.dump(imputed_data_store, f)
print("Saved 'imputed_data_store.json'")

# 2. Create Soil Type Store (which district maps to which soil_* columns)
soil_cols = [col for col in df_general.columns if col.startswith('soil_')]
soil_type_store = {}
for col in dist_cols:
    district_name = col.split('_', 1)[1]
    district_data = df_general[df_general[col] == 1].iloc[0]
    soil_type_store[district_name] = {s_col: int(district_data[s_col]) for s_col in soil_cols}
soil_type_store['default'] = {s_col: 0 for s_col in soil_cols}
soil_type_store['default']['soil_unknown'] = 1 # Default fallback

with open('soil_type_store.json', 'w') as f:
    json.dump(soil_type_store, f)
print("Saved 'soil_type_store.json'")

# 3. Create Weather Store (long-term averages per district/season)
# We use df_base which has the original season/district names
weather_store = df_base.groupby(['district', 'season'])[['avg_temp_c', 'total_precip_mm']].mean().to_dict('index')
# Convert tuples keys to strings for JSON
weather_store_json = {f"{k[0]}_{k[1]}": v for k, v in weather_store.items()}
weather_store_json['default'] = { # Global fallback
    'avg_temp_c': df_base['avg_temp_c'].mean(),
    'total_precip_mm': df_base['total_precip_mm'].mean()
}

with open('weather_store.json', 'w') as f:
    json.dump(weather_store_json, f)
print("Saved 'weather_store.json'")

Saved 'imputed_data_store.json'
Saved 'soil_type_store.json'
Saved 'weather_store.json'


In [3]:
from sklearn.ensemble import RandomForestRegressor
# Reload general data
df_general = pd.read_csv('features_v3_ohe_crop.csv')
df_general = df_general[df_general['crop_sugarcane'] == 0].copy()

# Drop sugarcane column (it's now zero-variance)
df_general = df_general.drop(columns=['crop_sugarcane'])

X_general = df_general.drop(columns=['yield_log1p'])
y_general = df_general['yield_log1p']

# Save the column list
general_model_columns = X_general.columns.tolist()
joblib.dump(general_model_columns, 'general_model_columns.pkl')
print("Saved 'general_model_columns.pkl'")

# Train the final model on ALL general crop data
rf_general_production = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_general_production.fit(X_general, y_general)
print("Trained final general model on all data.")

# Save the final model
joblib.dump(rf_general_production, 'general_yield_model.pkl')
print("Saved 'general_yield_model.pkl'")

Saved 'general_model_columns.pkl'
Trained final general model on all data.
Saved 'general_yield_model.pkl'
